# Knowledge Graph Exploration
Load the product knowledge graph and run graph analytics queries.

In [ ]:
import sys
sys.path.insert(0, '..')
import json
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML

graph_path = Path('../results/product_kg.graphml')
if graph_path.exists():
    G = nx.read_graphml(graph_path)
    print(f'Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    node_types = {}
    for n, d in G.nodes(data=True):
        t = d.get('type', 'Unknown')
        node_types[t] = node_types.get(t, 0) + 1
    print('Node types:', node_types)
else:
    print('Run the graph stage first.')

In [ ]:
# Find all products by a specific brand
if graph_path.exists():
    def find_products_by_brand(G, brand_name):
        brand_name = brand_name.lower()
        results = []
        for n, d in G.nodes(data=True):
            if d.get('type') == 'Brand' and brand_name in str(d.get('name', '')).lower():
                # Find products pointing to this brand
                for pred in G.predecessors(n):
                    pred_data = G.nodes[pred]
                    if pred_data.get('type') == 'Product':
                        results.append(pred_data.get('title', pred))
        return results

    products = find_products_by_brand(G, 'nike')
    print(f'Products by Nike: {len(products)}')
    for p in products[:5]:
        print(f'  - {p}')

In [ ]:
# Find VARIANT_OF clusters
if graph_path.exists():
    variant_edges = [(u, v, d) for u, v, d in G.edges(data=True) if d.get('relation') == 'VARIANT_OF']
    print(f'VARIANT_OF edges: {len(variant_edges)}')
    for u, v, d in variant_edges[:5]:
        t_u = G.nodes[u].get('title', u)[:60]
        t_v = G.nodes[v].get('title', v)[:60]
        print(f'  sim={d.get("confidence", "?")}: "{t_u}" <-> "{t_v}"')

In [ ]:
# Betweenness centrality on a subgraph (100 nodes for speed)
if graph_path.exists():
    degrees = dict(G.degree())
    top100 = sorted(degrees, key=lambda x: -degrees[x])[:100]
    sub = G.subgraph(top100)
    bc = nx.betweenness_centrality(sub)
    top_bc = sorted(bc.items(), key=lambda x: -x[1])[:10]
    print('Top 10 nodes by betweenness centrality:')
    for node_id, score in top_bc:
        nd = G.nodes[node_id]
        label = nd.get('name', nd.get('title', node_id))[:50]
        print(f'  {score:.4f} | {nd.get("type", "?")} | {label}')

In [ ]:
# Display interactive graph HTML (if available)
html_path = Path('../results/graph_visualization.html')
if html_path.exists():
    with open(html_path) as f:
        html_content = f.read()
    display(HTML('<p><b>Interactive Knowledge Graph</b> — see <a href="../results/graph_visualization.html" target="_blank">graph_visualization.html</a> for full interactive view.</p>'))
else:
    print('HTML visualization not found. Run the graph stage.')

In [ ]:
# Graph stats summary
stats_path = Path('../results/graph_stats.json')
if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)
    print('Knowledge Graph Summary:')
    print(f"  Total nodes: {stats['total_nodes']:,}")
    print(f"  Total edges: {stats['total_edges']:,}")
    print(f"  Average degree: {stats['average_degree']}")
    print(f"  Connected components: {stats['num_weakly_connected_components']}")
    print('\nTop brands by connections:')
    for brand, deg in stats.get('top_10_brands_by_degree', []):
        print(f'  {brand}: {deg} connections')